# County Obesity Hotspot Analysis Workflow

This notebook demonstrates a clean multi-agent GAS workflow for a real public-health use case:

Question: Where are the statistically significant clusters (hot spots and cold spots) of adult obesity prevalence across U.S. counties?

## Install GAS Client SDK

This notebook uses the published `gas-client` package from PyPI. Run this cell once in a new notebook environment.

In [ ]:
%pip install -q gas-client

## Imports

In [ ]:
from urllib.parse import urljoin
from IPython.display import HTML, Image, display
from gas_client import GasClient

## User Settings


In [ ]:

server_url = "http://127.0.0.1:4042"

openai_api_key = "YOUR_OPENAI_API_KEY"
timeout_seconds = 1800

client = GasClient(
    server_url,
    openai_api_key=openai_api_key,
    artifact_delivery="URL",
)

credentials = {"OPENAI_API_KEY": openai_api_key}

## Bind to the Four Agents

In [ ]:
data_agent = client.agent("geospatial_data_retrieval_agent")
vector_agent = client.agent("vector_analysis_agent")
stats_agent = client.agent("spatial_statistics_agent")
mapping_agent = client.agent("mapping_agent")

for agent in [data_agent, vector_agent, stats_agent, mapping_agent]:
    print(agent.agent_id, agent.status().get("status"))

## Helper Functions


In [4]:
def absolute_url(url):
    if url.startswith("/"):
        return urljoin(server_url, url)
    return url


def first_artifact_url(task_result, preferred_extensions=None):
    artifacts = task_result.get("outputs", {}).get("artifacts", [])
    preferred_extensions = preferred_extensions or []

    for extension in preferred_extensions:
        for artifact in artifacts:
            url = artifact.get("url")
            filename = artifact.get("filename") or artifact.get("name") or url or ""
            if url and str(filename).lower().endswith(extension.lower()):
                return absolute_url(url)

    for artifact in artifacts:
        if artifact.get("url"):
            return absolute_url(artifact["url"])

    raise RuntimeError("The task returned no artifact URL.")


def display_visual_artifacts(task_result):
    artifacts = task_result.get("outputs", {}).get("artifacts", [])
    for artifact in artifacts:
        url = artifact.get("url")
        filename = artifact.get("filename") or artifact.get("name") or ""
        if not url:
            continue

        display_url = absolute_url(url)
        lower_name = str(filename).lower()
        if lower_name.endswith(".png"):
            display(Image(url=display_url))
        elif lower_name.endswith(".html"):
            display(HTML(f'<a href="{display_url}" target="_blank">Open HTML artifact: {filename}</a>'))
        else:
            display(HTML(f'<a href="{display_url}" target="_blank">Open artifact: {filename}</a>'))


def run_streaming_task(agent, instructions, *, input_datasets=None, title=None):
    if title:
        print("\n" + "=" * 80)
        print(title)
        print("=" * 80)

    final_result = None
    for event in agent.execute_task(
        instructions,
        mode="stream",
        input_datasets=input_datasets,
        artifact_delivery="URL",
        credentials=credentials,
        timeout=timeout_seconds,
    ):
        client.print_stream_event(event)
        if event.get("event") == "task_result":
            final_result = event.get("payload")

    if final_result is None:
        raise RuntimeError("The stream ended before returning a task_result event.")

    client.print_task_summary(final_result)
    return final_result

## Step 1: Download County Boundaries


In [ ]:
boundary_result = run_streaming_task(
    data_agent,
    (
        "Use the U.S. Census Bureau TIGER boundary source. Download 2020 county polygons for the "
        "contiguous United States. Exclude Alaska, Hawaii, Puerto Rico, and other territories. "
        "Return one clean GeoPackage dataset with county geometry, GEOID, county name, state name "
        "or abbreviation, STATEFP, and COUNTYFP fields. No source API key is required."
    ),
    title="Download Contiguous US County Boundaries",
)

county_boundary_url = first_artifact_url(
    boundary_result, preferred_extensions=[".gpkg", ".geojson", ".json"]
)
county_boundary_url

## Step 2: Download CDC PLACES Obesity Prevalence


In [ ]:
obesity_result = run_streaming_task(
    data_agent,
    (
        "Download 2020 county-level crude prevalence of adult "
        "obesity for the contiguous United States. Return one clean dataset with county GEOID/FIPS, "
        "county name, state identifier, year, measure name, and the obesity prevalence value as a "
        "numeric column. Geometry is not required for this dataset because it will be joined to a "
        "separate county boundary dataset. No source API key is required."
    ),
    title="Download 2020 CDC PLACES County Obesity Prevalence",
)

obesity_url = first_artifact_url(
    obesity_result, preferred_extensions=[".csv", ".gpkg", ".geojson", ".json"]
)
obesity_url

## Step 3: Join Boundaries With Obesity Prevalence


In [ ]:
join_result = run_streaming_task(
    vector_agent,
    (
        "Join the contiguous US county boundary dataset with the 2020 CDC PLACES county-level adult "
        "obesity prevalence dataset. Use GEOID or county FIPS as the join key (pad to 5 characters if "
        "needed so STATEFP + COUNTYFP matches the FIPS string). Preserve county boundary geometry and "
        "key identifiers. Rename the obesity prevalence column to a clean numeric field named "
        "'obesity_rate'. Keep county name, state, GEOID, year, and measure fields when available. "
        "Drop counties with missing obesity_rate. Return one GeoPackage ready for spatial statistics."
    ),
    input_datasets=[county_boundary_url, obesity_url],
    title="Join County Boundaries With CDC Obesity Prevalence",
)

county_obesity_url = first_artifact_url(
    join_result, preferred_extensions=[".gpkg", ".geojson", ".json"]
)
county_obesity_url

## Step 4: Local Moran's I (LISA) Cluster Analysis


In [ ]:
lisa_result = run_streaming_task(
    stats_agent,
    (
        "Perform an Exploratory Spatial Data Analysis on the input county dataset using PySAL. "
        "Target variable: obesity_rate. "
        "1) Build Queen-contiguity spatial weights and row-standardize them. "
        "2) Compute the Global Moran's I statistic with 999 permutations and report the I value, "
        "expected I, z-score, and pseudo p-value. "
        "3) Compute Local Moran's I (LISA) with 999 permutations and classify each county into "
        "'High-High', 'Low-Low', 'High-Low', 'Low-High', or 'Not Significant' at p < 0.05. "
        "4) Return a polished HTML report with the Moran scatterplot, a LISA significance map, and "
        "a LISA cluster map embedded. "
        "5) Also return a GeoPackage of the input counties augmented with the new fields "
        "'local_I', 'local_p_value', and 'lisa_cluster' (the categorical label). "
        "Interpret the results in the report in plain English suitable for a public-health audience."
    ),
    input_datasets=[county_obesity_url],
    title="Local Moran's I (LISA) Hotspot Analysis",
)

lisa_cluster_url = first_artifact_url(
    lisa_result, preferred_extensions=[".gpkg", ".geojson", ".json"]
)
lisa_cluster_url

### View the Statistical Report


In [ ]:
display_visual_artifacts(lisa_result)